# RiskBands API UX Demo

Este notebook mostra a camada mais amigável da API do RiskBands no estilo sklearn/pandas.

Objetivos da demo:

- começar com um fluxo curto e memorável
- usar `DataFrame` com `y="target"`
- comparar `legacy` e `stable`
- inspecionar `summary()`, `binning_table()`, `score_details()` e `diagnostics()`
- gerar visuais comparativos com Plotly a partir de dados sintéticos


In [1]:
import plotly

print(f"Plotly version: {plotly.__version__}")


Plotly version: 6.7.0


In [2]:
# %cd D:/0_CienciaDados/1_Frameworks/RiskBands
# ! pip install -r requirements.txt
# ! pip install -e .

In [3]:
from riskbands import Binner
import inspect
print(inspect.signature(Binner.__init__))

(self, strategy: 'str' = 'supervised', max_bins: 'int' = 6, max_n_bins: 'int | None' = None, min_event_rate_diff: 'float' = 0.02, monotonic: 'str | None' = None, monotonic_trend: 'str | None' = None, check_stability: 'bool' = False, use_optuna: 'bool' = False, time_col: 'str | None' = None, force_categorical: 'list[str] | None' = None, force_numeric: 'list[str] | None' = None, score_strategy: 'str' = 'legacy', score_weights: 'dict | None' = None, normalization_strategy: 'str' = 'absolute', woe_shrinkage_strength: 'float' = 25.0, objective_kwargs: 'dict | None' = None, strategy_kwargs: 'dict | None' = None)


In [4]:
import inspect
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display


def _find_repo_root(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / 'riskbands' / 'binning_engine.py').exists():
            return base
    return None


repo_root = _find_repo_root()
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Remove qualquer versão já carregada do pacote, para evitar usar uma instalação antiga.
for module_name in list(sys.modules):
    if module_name == 'riskbands' or module_name.startswith('riskbands.'):
        del sys.modules[module_name]

import riskbands
from riskbands import Binner

binner_signature = inspect.signature(Binner.__init__)
required_params = {'score_strategy', 'max_n_bins'}
missing_params = required_params - set(binner_signature.parameters)

print('riskbands importado de:', getattr(riskbands, '__file__', 'desconhecido'))
print('Assinatura de Binner.__init__:', binner_signature)

if missing_params:
    raise RuntimeError(
        'O notebook ainda está importando uma versão antiga do pacote RiskBands. '
        f'Parâmetros ausentes: {sorted(missing_params)}. '
        'Rode o notebook a partir do repositório atualizado, reinstale o pacote em modo editável '
        'ou ajuste o PYTHONPATH para apontar para a versão nova.'
    )


riskbands importado de: D:\0_CienciaDados\1_Frameworks\RiskBands\riskbands\__init__.py
Assinatura de Binner.__init__: (self, strategy: 'str' = 'supervised', max_bins: 'int' = 6, max_n_bins: 'int | None' = None, min_event_rate_diff: 'float' = 0.02, monotonic: 'str | None' = None, monotonic_trend: 'str | None' = None, check_stability: 'bool' = False, use_optuna: 'bool' = False, time_col: 'str | None' = None, force_categorical: 'list[str] | None' = None, force_numeric: 'list[str] | None' = None, score_strategy: 'str' = 'legacy', score_weights: 'dict | None' = None, normalization_strategy: 'str' = 'absolute', woe_shrinkage_strength: 'float' = 25.0, objective_kwargs: 'dict | None' = None, strategy_kwargs: 'dict | None' = None)


> **Nota importante**
>
> Se esta célula acusar que `score_strategy` ou `max_n_bins` estão ausentes, o problema não está neste notebook.
> Nesse caso, o kernel está importando uma versão antiga do pacote `riskbands` instalada no ambiente, em vez da versão atual do repositório.


In [5]:
rng = np.random.default_rng(42)
n = 10000

df = pd.DataFrame(
    {
        "score": rng.normal(loc=0.0, scale=1.0, size=n),
        "utilization": rng.beta(2.5, 4.0, size=n),
        "month": rng.choice([202301, 202302, 202303, 202304, 202305, 202306], size=n),
    }
)

month_drift = (df["month"] - df["month"].min()) / 100
logit = -1.2 + 1.1 * df["score"] + 1.8 * df["utilization"] + 0.35 * month_drift
prob = 1.0 / (1.0 + np.exp(-logit))
df["target"] = (rng.random(n) < prob).astype(int)

df.head()


,score,utilization,month,target
0,0.304717,0.543599,202304,0
1,-1.039984,0.104586,202301,0
2,0.750451,0.798964,202303,1
3,0.940565,0.452367,202303,1
4,-1.951035,0.225358,202301,0


## 1. Fit no estilo dataframe-first

O fluxo recomendado agora é curto:

1. instanciar o `Binner`
2. chamar `fit(df, y="target", column="score", time_col="month")`
3. inspecionar `summary()` e `score_details()`
4. aplicar `transform(...)` quando quiser anexar o resultado ao `DataFrame`


In [6]:
legacy_binner = Binner(
    strategy="supervised",
    max_n_bins=5,
    check_stability=True,
    score_strategy="legacy",
)

stable_binner = Binner(
    strategy="supervised",
    max_n_bins=5,
    check_stability=True,
    score_strategy="stable",
    normalization_strategy="absolute",
    woe_shrinkage_strength=35.0,
)

legacy_binner.fit(df, y="target", column="score", time_col="month")
stable_binner.fit(df, y="target", column="score", time_col="month")

friendly_summary = pd.concat(
    [
        legacy_binner.summary().assign(run="legacy"),
        stable_binner.summary().assign(run="stable"),
    ],
    ignore_index=True,
)
friendly_summary


d:\0_CienciaDados\1_Frameworks\RiskBands\.venv-pypi\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,variable,n_bins,iv,temporal_score,score_strategy,objective_direction,objective_score,objective_preference_score,coverage_ratio_min,coverage_ratio_mean,rare_bin_count,ranking_reversal_period_count,selection_basis,alert_flags,run
0,score,5,0.863726,0.0,legacy,maximize,0.317951,0.317951,0.0,0.0,0,0,discrimination-led,low_coverage,legacy
1,score,5,0.863726,0.0,stable,minimize,0.280000,-0.280000,0.0,0.0,0,0,no_objective_signal,low_coverage,stable


In [7]:
score_bins = stable_binner.transform(df["score"])
df_preview = df.assign(score_bin=score_bins)

binning_table = stable_binner.binning_table()
score_details = stable_binner.score_details()
temporal_diagnostics = stable_binner.diagnostics(df, y="target", time_col="month", kind="bin")

display(df_preview.head())
display(binning_table)
display(score_details)
display(temporal_diagnostics.head())


,score,utilization,month,target,score_bin
0,0.304717,0.543599,202304,0,-0.195163
1,-1.039984,0.104586,202301,0,1.660272
2,0.750451,0.798964,202303,1,-1.005740
3,0.940565,0.452367,202303,1,-1.005740
4,-1.951035,0.225358,202301,0,1.660272


,variable,bin,count,event,non_event,event_rate
0,score,"(-inf, -0.97)",1730,191,1539,0.110405
1,score,"[-0.97, -0.05)",3125,862,2263,0.275840
2,score,"[-0.05, 0.65)",2633,1165,1468,0.442461
3,score,"[0.65, 1.51)",1852,1187,665,0.640929
4,score,"[1.51, inf)",660,545,115,0.825758


,variable,score_strategy,objective_direction,objective_score,objective_preference_score,objective_base_score,objective_total_penalty,objective_normalization_strategy,woe_shrinkage_strength,objective_raw_iv,...,objective_norm_separation_reward,objective_norm_separation_penalty,objective_norm_entropy,objective_norm_psi,objective_weight_temporal_variance_weight,objective_weight_window_drift_weight,objective_weight_rank_inversion_weight,objective_weight_separation_weight,objective_weight_entropy_weight,objective_weight_psi_weight
0,score,stable,minimize,0.28,-0.28,0.0,0.28,absolute,35.0,0.863726,...,0.0,1.0,1.0,0.0,0.22,0.18,0.2,0.2,0.08,0.12


,dataset,variable,bin_code,bin,bin_order,month,total_count,event_count,non_event_count,bin_share,...,rare_bin_flag,missing_bin_flag,low_coverage_flag,period_monotonic_break_flag,period_ranking_reversal_flag,expected_trend,alert_flags,min_bin_count,min_bin_share,min_time_coverage
0,None,score,0.0,"(-inf, -0.97)",0,202301,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
1,None,score,0.0,"(-inf, -0.97)",0,202302,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
2,None,score,0.0,"(-inf, -0.97)",0,202303,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
3,None,score,0.0,"(-inf, -0.97)",0,202304,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
4,None,score,0.0,"(-inf, -0.97)",0,202305,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75


## 2. Comparação visual com Plotly

Aqui a ideia é mostrar que a API amigável não esconde a arquitetura nova de scoring:

- `legacy` continua disponível
- `stable` continua controlável
- ambos podem ser comparados no mesmo fluxo de notebook


In [8]:
comparison = pd.concat(
    [
        legacy_binner.score_details().assign(run="legacy"),
        stable_binner.score_details().assign(run="stable"),
    ],
    ignore_index=True,
).copy()

comparison_table = (
    comparison[
        [
            "run",
            "score_strategy",
            "objective_direction",
            "objective_score",
            "objective_preference_score",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "run": "strategy",
            "score_strategy": "score_strategy",
            "objective_direction": "direction",
            "objective_score": "raw_score",
            "objective_preference_score": "internal_comparison_score",
        }
    )
    .reset_index(drop=True)
)

display(comparison_table)

raw_plot = comparison_table.copy()
raw_plot["text_label"] = raw_plot.apply(
    lambda row: f"raw={row['raw_score']:.3f}<br>{row['direction']}",
    axis=1,
)

fig_raw = px.bar(
    raw_plot,
    x="strategy",
    y="raw_score",
    color="direction",
    text="text_label",
    title="Raw objective score by strategy",
    hover_data={
        "score_strategy": True,
        "direction": True,
        "raw_score": ":.4f",
        "internal_comparison_score": ":.4f",
        "text_label": False,
    },
)

fig_raw.update_traces(textposition="outside")
fig_raw.update_layout(
    xaxis_title="strategy",
    yaxis_title="raw objective score",
)
fig_raw


,strategy,score_strategy,direction,raw_score,internal_comparison_score
0,legacy,legacy,maximize,0.317951,0.317951
1,stable,stable,minimize,0.280000,-0.280000


### Como ler a comparação

- `raw_score` deve ser lido **dentro da lógica de cada estratégia**.
- Em `legacy`, o objetivo é **maximizar** o score bruto.
- Em `stable`, o objetivo é **minimizar** o score bruto.
- Portanto, esse gráfico **não deve ser lido como disputa direta de vencedor** entre estratégias.
- Para comparação mais justa, olhe também os componentes, a estabilidade temporal e a tabela de bins.


In [9]:
temporal_diagnostics = stable_binner.diagnostics(
    df,
    y="target",
    time_col="month",
    kind="bin",
)

display(temporal_diagnostics.head())
print("Shape:", temporal_diagnostics.shape)
print("Linhas com total_count > 0:", int((temporal_diagnostics["total_count"] > 0).sum()))


,dataset,variable,bin_code,bin,bin_order,month,total_count,event_count,non_event_count,bin_share,...,rare_bin_flag,missing_bin_flag,low_coverage_flag,period_monotonic_break_flag,period_ranking_reversal_flag,expected_trend,alert_flags,min_bin_count,min_bin_share,min_time_coverage
0,None,score,0.0,"(-inf, -0.97)",0,202301,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
1,None,score,0.0,"(-inf, -0.97)",0,202302,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
2,None,score,0.0,"(-inf, -0.97)",0,202303,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
3,None,score,0.0,"(-inf, -0.97)",0,202304,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75
4,None,score,0.0,"(-inf, -0.97)",0,202305,0,0,0,0.0,...,False,True,True,False,False,none,missing_bin;low_coverage,30,0.05,0.75


Shape: (30, 27)
Linhas com total_count > 0: 0


In [10]:
stability_pivot = stable_binner.stability_over_time(
    df[["score", "month"]],
    df["target"],
    time_col="month",
)

display(stability_pivot.head())

# Mantém apenas a variável principal desta demo
if "variable" in stability_pivot.index.names:
    heatmap_df = stability_pivot.xs("score", level="variable").copy()
else:
    heatmap_df = stability_pivot.copy()

# Remove bins totalmente vazios, caso existam
heatmap_df = heatmap_df.dropna(how="all")

if heatmap_df.empty:
    raise ValueError(
        "O pivot de estabilidade ficou vazio. Verifique se `score` e `month` foram passados corretamente para `stability_over_time(...)`."
    )

fig_heatmap = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Event rate by bin and month",
    labels={"x": "month", "y": "bin", "color": "event_rate"},
)

fig_heatmap.update_layout(
    xaxis_title="month",
    yaxis_title="bin",
)

fig_heatmap


month                 202301    202302    202303    202304    202305    202306
variable bin                                                                  
score    -1.982196  0.796610  0.820755  0.789474  0.846847  0.858407  0.837607
         -1.005740  0.659016  0.629393  0.623377  0.651316  0.661392  0.620915
         -0.195163  0.440639  0.469136  0.435268  0.437642  0.421397  0.453725
          0.538849  0.275335  0.280392  0.264706  0.262760  0.298851  0.273070
          1.660272  0.093633  0.113565  0.120521  0.113402  0.108527  0.110345

### Nota sobre o heatmap

Para este gráfico, o notebook usa `stability_over_time(...)` em vez da tabela completa de `diagnostics()`.
Isso deixa a visualização mais robusta e mais aderente ao objetivo do gráfico: mostrar a taxa de evento por bin ao longo do tempo.
A tabela de `diagnostics()` continua útil para auditoria detalhada, flags e investigação de cobertura.


In [11]:
component_columns = [
    column
    for column in score_details.columns
    if column.startswith("objective_norm_")
]
component_frame = score_details.melt(
    id_vars=["variable", "score_strategy"],
    value_vars=component_columns,
    var_name="component",
    value_name="value",
)

fig_components = px.bar(
    component_frame,
    x="component",
    y="value",
    color="score_strategy",
    barmode="group",
    title="Normalized stable objective components",
)
fig_components.update_layout(xaxis_tickangle=-35)
fig_components


## 3. Leituras rápidas

- `summary()` é a melhor porta de entrada para entender o resultado logo após o `fit`
- `binning_table()` deixa os cortes fáceis de inspecionar no notebook
- `score_details()` expõe o score final, os componentes e os pesos do objective
- `diagnostics(kind="bin")` ajuda a abrir estabilidade temporal sem vasculhar estruturas internas
- a camada pública ficou mais familiar, mas o controle fino de `score_strategy`, `score_weights`, `normalization_strategy` e `woe_shrinkage_strength` continua disponível
